In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re # We use regular expressions for data cleaning
import warnings
warnings.filterwarnings('ignore')

import nltk
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score,f1_score, roc_auc_score, roc_curve, # evaluatin metrics
                             confusion_matrix,classification_report)
import tensorflow as tf
import transformers
from transformers import TFAutoModel, AutoTokenizer
from nltk.corpus import stopwords
from sklearn.linear_model import LogisticRegression


plt.style.use('seaborn')

# New Section

In [ ]:
df1= pd.read_excel('egyptian_dataset.xlsx',header=0)
df1

FileNotFoundError: [Errno 2] No such file or directory: 'egyptian_dataset.xlsx'

In [ ]:
df2= pd.read_csv("Reviews.csv").drop('Unnamed: 0',axis=1)
df2

In [ ]:
df=pd.concat([df1,df2],axis=0)
df.shape

In [ ]:
import matplotlib.pyplot as plt

# Get label distribution
labels_distribution = df['data_labels'].value_counts()

# Plot a pie chart
plt.figure(figsize=(8, 8))
plt.pie(labels_distribution, labels=labels_distribution.index, autopct='%1.1f%%', colors=['#66b3ff','#99ff99'])
plt.title('Distribution of Labels (0 and 1)')
plt.show()

# Print the counts
print("Label distribution:")
print(labels_distribution)


In [ ]:

def data_cleaning(text):
    """Clean and preprocess text data.
    Args:
        text (pd.Series): A pandas Series containing text data to be cleaned.
    Returns:
        pd.Series: A pandas Series with the cleaned text data.

    Cleaning Steps:
    - Removes emojis and special characters like '\x89Û_', '&amp', etc.
    - Replaces consecutive dots with an empty string.
    - Removes '#' symbol from text.
    - Removes user names starting with '@'.
    - Removes URLs starting with 'http' or 'https'.
    - Converts text to lowercase.
    - Removes extra whitespaces between words.

    """
    clean = text
    # Replace consecutive dots with an empty string
    pattern = re.compile('\\.+?(?=\B|$)')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=''))
    # Replace '\x89Û_' with a whitespace
    pattern = re.compile('\x89Û_')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=' '))
    # Replace '&amp' with '&'
    pattern = re.compile('\&amp')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=' '))
     # Replace newline characters with a whitespace
    pattern = re.compile('\\n')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=' '))
    # Remove '#' symbol from text
    clean = clean.apply(lambda r: r.replace('#', ''))
    # Replace user names with '@'
    pattern = re.compile('@[a-zA-Z0-9\_]+')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl='@'))
    # Remove URLs
    pattern = re.compile('https?\S+(?=\s|$)')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl='www'))
    # Convert text to lowercase
    clean = clean.apply(lambda r: r.lower())
    # Remove extra whitespaces
    clean = clean.apply(lambda r: ' '.join(r.split()))  # Remove extra whitespaces between words

    return clean


In [ ]:
df['texts'] = data_cleaning(df['texts'])


In [ ]:
tweets = df['texts']
labels = df['data_labels']

X_train, X_test, y_train, y_test = train_test_split(tweets, labels,stratify=labels, test_size=0.15, random_state=1)

In [ ]:
def preprocess_input_data(texts, tokenizer, max_len=120):
    """Tokenize and preprocess the input data for Arabert model.

    Args:
        texts (list): List of text strings.
        tokenizer (AutoTokenizer): Arabert tokenizer from transformers library.
        max_len (int, optional): Maximum sequence length. Defaults to 120.

    Returns:
        Tuple of numpy arrays: Input token IDs and attention masks.
    """
    # Tokenize the text data using the tokenizer
    tokenized_data = [tokenizer.encode_plus(
        t,
        max_length=max_len,
        pad_to_max_length=True,
        add_special_tokens=True) for t in texts]

    # Extract tokenized input IDs and attention masks
    input_ids = [data['input_ids'] for data in tokenized_data]
    attention_mask = [data['attention_mask'] for data in tokenized_data]

    return input_ids, attention_mask

In [ ]:
# Download Arabert tokenizer
arabert_tokenizer = AutoTokenizer.from_pretrained('aubmindlab/bert-base-arabert')

# Preprocess the training data
X_train_input_ids, X_train_attention_mask = preprocess_input_data(X_train, arabert_tokenizer)


In [ ]:
'''# Calculate the confusion matrix
conf_matrix = confusion_matrix(y_test, predictions)

# Visualize the confusion matrix using a heatmap
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Class 0', 'Class 1'], yticklabels=['Class 0', 'Class 1'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()'''

In [ ]:
def create_arabert(model_name):
    """Create a classification model using AraBERT.

    Args:
        model_name (str): Name of the pre-trained AraBERT model to use.

    Returns:
        tf.keras.Model: Compiled classification model.

    This function creates a classification model based on AraBERT. It takes the model_name,
    initializes the AraBERT model, and then adds a classification head on top of it.

    The classification head consists of a Dense layer with sigmoid activation, which is
    suitable for binary classification tasks. The model is compiled with the Adam optimizer
    and binary cross-entropy loss.

    """
    # Define token ids as inputs
    word_inputs = tf.keras.Input(shape=(120,), name='word_inputs', dtype='int32')

    # Call AraBERT model
    arabert = TFAutoModel.from_pretrained(model_name)
    input_ids = word_inputs
    attention_mask = tf.where(input_ids != 0, 1, 0)  # Creating attention mask
    arabert_encodings = arabert(input_ids, attention_mask=attention_mask)[0]

    # CLASSIFICATION HEAD
    # Collect last hidden state (CLS)
    doc_encoding = tf.squeeze(arabert_encodings[:, 0:1, :], axis=1)
    # Apply dropout for regularization
    doc_encoding = tf.keras.layers.Dropout(.1)(doc_encoding)
    # Final output
    outputs = tf.keras.layers.Dense(1, activation='sigmoid', name='outputs')(doc_encoding)

    # Compile model
    model = tf.keras.Model(inputs=[word_inputs], outputs=[outputs])

    # Optional: Set class weights if dealing with imbalanced data

    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5), loss='binary_crossentropy',
                      metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])
    return model


In [ ]:
arabert_model = 'aubmindlab/bert-base-arabert'
arabert = create_arabert(arabert_model)

In [ ]:
tf.random.set_seed(3)
class_weights = {0: 1, 1: 5}
hist = arabert.fit(
    x=X_train_input_ids,
    y=y_train.to_list(),
    epochs=3,
    batch_size=16
)

In [ ]:
def roc_curve__confusion_matrix(model):
    from sklearn.metrics import roc_curve, auc
    inp_tok, ids = preprocess_input_data(X_test, arabert_tokenizer)
    y_prob_test = model.predict(inp_tok, verbose=True)
    inp_tok_train, ids_train = preprocess_input_data(X_train,arabert_tokenizer)
    y_prob_train = model.predict(inp_tok_train, verbose=True)


    prediction=np.where(y_prob_test <0.5,0,1).reshape(len(X_test),1)
    # calculate tpr ,fpr
    fpr_test , tpr_test , thresholds = roc_curve(y_test, y_prob_test)
    fpr_train , tpr_train , thresholds1 = roc_curve(y_train, y_prob_train)

    # calculate AUC score
    roc_auc_test  = auc(fpr_test, tpr_test)
    roc_auc_train  = auc(fpr_train ,tpr_train)


    # calculate confusion matrix for test set
    confusion_matrix_test = confusion_matrix(y_test, prediction)

    # create subplots
    fig, axs = plt.subplots(1, 2, figsize=(12, 6))

    # plot ROC curves on the first subplot
    axs[0].plot(fpr_train, tpr_train, color='black', label='ROC curve train (AUC = %0.2f)' % roc_auc_train)
    axs[0].plot(fpr_test, tpr_test, color='green', label='ROC curve test (AUC = %0.2f)' % roc_auc_test)
    axs[0].plot([0, 1], [0, 1], color='red', linestyle='--')
    axs[0].set_xlabel('FP Rate')
    axs[0].set_ylabel('TP Rate')
    axs[0].set_title('Receiver Operating Characteristic (ROC) Curve')
    axs[0].legend(loc="lower right")

    sns.set(font_scale=1.4)
    sns.heatmap(confusion_matrix_test, annot=True, fmt='g', cmap='Blues', ax=axs[1])
    axs[1].set_xlabel('Predicted label')
    axs[1].set_ylabel('True label')
    axs[1].set_title('Confusion Matrix');

    plt.tight_layout()
    plt.show()

roc_curve__confusion_matrix(arabert)

In [ ]:
X_test_input_ids, X_test_attention_mask = preprocess_input_data(X_test, arabert_tokenizer)
y_prob_test = arabert.predict(X_test_input_ids, verbose=True)
prediction=np.where(y_prob_test <0.5,0,1).reshape(len(X_test),1)
print(accuracy_score(prediction,y_test))
print(f1_score(prediction,y_test))

In [ ]:
test=["ابشرووو","ما فهمتش عليه","ما ننصحش بيه ","ما يستاهلش","خيبة أمل كبيرة ومخيبة","جودة ضعيفة","good service","مشاكل كثيرة","بطء في التوصيل","خدمة جيدة",
      "produit super","gamiil awiii","عجبني اوييي","يعطيك صحة "]
inp_tok, ids= preprocess_input_data(test, arabert_tokenizer)
preds = arabert.predict(inp_tok, verbose=True)
predicted_class=list(np.where(preds <0.5,0,1).reshape(len(preds),1))
pd.DataFrame({"text":test,"prediction":list(preds),"predicted_class":predicted_class})